# 🧭 Graph Traversal — Runnable Notebook

Companion to [`README.md`](README.md) and
[`07_graph_traversal_lesson.html`](07_graph_traversal_lesson.html).

**BFS** (queue, shortest paths) and **DFS** (stack/recursion, structure) — plus components and cycle detection.

## 1. Build the graph (adjacency list)
We sort neighbours so traversal order is deterministic for the demos.

In [ ]:
from collections import deque, defaultdict

def build_graph(edges, directed=False):
    adj = defaultdict(list)
    for u, v in edges:
        adj[u].append(v)
        if not directed:
            adj[v].append(u)
    for u in adj:
        adj[u].sort()                 # deterministic order for reproducible demos
    return adj

edges = [("A","B"), ("A","D"), ("B","C"), ("B","E"), ("C","F"), ("D","E"), ("E","F")]
adj = build_graph(edges)
print({k: adj[k] for k in sorted(adj)})

## 2. BFS — visit order AND shortest distance
The first time BFS reaches a vertex is via a fewest-edges path.

In [ ]:
def bfs(adj, start):
    """BFS from `start`: returns visit order and shortest distance (in edges)."""
    seen = {start}                    # mark on ENQUEUE so nothing is queued twice
    dist = {start: 0}
    q = deque([start])
    order = []
    while q:
        u = q.popleft()               # oldest waiting vertex (FIFO -> wide)
        order.append(u)
        for v in adj[u]:
            if v not in seen:
                seen.add(v)
                dist[v] = dist[u] + 1  # one edge further than u
                q.append(v)
    return order, dist

order, dist = bfs(adj, "A")
print("BFS order:", order)
print("distances:", dist)
assert order == ["A", "B", "D", "C", "E", "F"]
assert dist["F"] == 3 and dist["B"] == 1

## 3. DFS — recursive and iterative (same result)

In [ ]:
def dfs_recursive(adj, start):
    """Dive into the first unvisited neighbour before trying the rest."""
    seen, order = set(), []
    def go(u):
        seen.add(u)
        order.append(u)
        for v in adj[u]:
            if v not in seen:
                go(v)
    go(start)
    return order

def dfs_iterative(adj, start):
    """Same walk with an explicit stack (safe for very deep graphs)."""
    seen, order, stack = set(), [], [start]
    while stack:
        u = stack.pop()               # LIFO -> depth-first
        if u in seen:
            continue
        seen.add(u)
        order.append(u)
        for v in reversed(adj[u]):    # reversed so the smallest neighbour pops first
            if v not in seen:
                stack.append(v)
    return order

print("DFS recursive:", dfs_recursive(adj, "A"))
print("DFS iterative:", dfs_iterative(adj, "A"))
assert dfs_recursive(adj, "A") == ["A", "B", "C", "F", "E", "D"]
assert dfs_iterative(adj, "A") == dfs_recursive(adj, "A")

## 4. Connected components (count the islands)
Start a fresh traversal from every not-yet-visited vertex.

In [ ]:
def connected_components(adj, vertices):
    """Each fresh start that finds new ground is one more component."""
    seen, count = set(), 0
    for start in vertices:
        if start not in seen:
            count += 1                # a new island
            stack = [start]
            while stack:              # flood the whole component
                u = stack.pop()
                if u in seen:
                    continue
                seen.add(u)
                stack.extend(adj[u])
    return count

print("components (connected graph):", connected_components(adj, sorted(adj)))
adj2 = build_graph(edges + [("G", "H")])          # add a separate island G-H
print("components (after adding G-H):", connected_components(adj2, sorted(adj2)))
assert connected_components(adj, sorted(adj)) == 1
assert connected_components(adj2, sorted(adj2)) == 2

## 5. Cycle detection (undirected)
DFS: reaching an already-visited vertex that isn't the one we came from = a cycle.

In [ ]:
def has_cycle_undirected(adj, vertices):
    """DFS tracking the parent; a visited non-parent neighbour means a cycle."""
    seen = set()
    def dfs(u, parent):
        seen.add(u)
        for v in adj[u]:
            if v not in seen:
                if dfs(v, u):
                    return True
            elif v != parent:         # visited AND not where we came from -> cycle
                return True
        return False
    for s in vertices:
        if s not in seen and dfs(s, None):
            return True
    return False

tree = build_graph([("A","B"), ("B","C"), ("B","D")])   # a tree = no cycle
print("our graph has a cycle?", has_cycle_undirected(adj, sorted(adj)))
print("a tree has a cycle?   ", has_cycle_undirected(tree, sorted(tree)))
assert has_cycle_undirected(adj, sorted(adj))
assert not has_cycle_undirected(tree, sorted(tree))

## ✅ Recap
- The **visited set** is what makes graph traversal safe (cycles!) and `O(V+E)`.
- **BFS = queue = wide = shortest path** (unweighted); **DFS = stack = deep = structure**.
- Components = traverse from **every unvisited vertex**.
- Cycle detection, topological sort, and bipartite checks are all "a traversal + a little bookkeeping".

That closes the tree + graph deep dive. See [`README`](../README.md) for the whole map.